In [ ]:
import requests, zipfile, os
import pandas as pd
import numpy as np
from itertools import combinations
from tabulate import tabulate

In [ ]:
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"

# Remove any old/corrupted file
if os.path.exists(zip_path):
    os.remove(zip_path)
    print("Removed old zip file.")

# Download with progress
print("Downloading fresh copy...")
response = requests.get(url, stream=True)
response.raise_for_status()  # Will raise error if URL is wrong

with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print(f"Downloaded: {os.path.getsize(zip_path)/1e6:.2f} MB")

# Verify it's a real ZIP
with zipfile.ZipFile(zip_path) as z:
    print("Valid ZIP! Contains:", z.namelist()[:5])
    print("Extracting...")
    z.extractall(".")

Downloaded: 0.98 MB
Valid ZIP! Contains: ['ml-latest-small/', 'ml-latest-small/links.csv', 'ml-latest-small/tags.csv', 'ml-latest-small/ratings.csv', 'ml-latest-small/README.txt']
Extracting...


In [ ]:
# Load ratings file
ratings = pd.read_csv(
    "ml-latest-small/ratings.csv",
    sep=",",
    encoding="latin-1"
)

# Show first few rows
print("\n✅ First five rows:")
print(ratings.head())

# Compute dataset summary
num_ratings = len(ratings)
num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()

print(f"\n📊 Dataset Summary:")
print(f"Row Count for Ratings Dataset: {num_ratings}")
print(f"Number of Users: {num_users}")
print(f"Number of Movies: {num_movies}")


✅ First five rows:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

📊 Dataset Summary:
Row Count for Ratings Dataset: 100836
Number of Users: 610
Number of Movies: 9724


In [ ]:
# Create a user-item matrix
def create_user_item_matrix(ratings):
    return ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)

user_item_matrix = create_user_item_matrix(ratings)

# Generate group item scores (mean rating for each item by the group)
def compute_group_item_scores(user_item_matrix, group_users):
    """
    Compute the average rating for items across a group of users.
    """
    group_data = user_item_matrix.loc[group_users]
    return group_data.mean(axis=0)

# Generate group recommendations (top N items recommended for the group)
def generate_group_recommendations(group_scores, n_items=10):
    """
    Generate group recommendations by selecting the top N items based on group scores.
    """
    return group_scores.sort_values(ascending=False).index[:n_items]

In [ ]:
# Counterfactual explanations
def generate_counterfactual_explanations(group_scores, user_item_matrix, group_users, n_items=10):
    top_items = group_scores.sort_values(ascending=False).index[:n_items]
    counterfactuals = {}

    # Interacted items
    interacted_items = user_item_matrix.columns[(user_item_matrix > 0).any()].tolist()

    for item in top_items:
        for size in range(1, len(interacted_items) + 1):
            for items_to_remove in combinations(interacted_items, size):

                modified_matrix = user_item_matrix.copy()
                modified_matrix[list(items_to_remove)] = 0

                modified_scores = modified_matrix.loc[group_users].mean(axis=0)
                new_recs = generate_group_recommendations(modified_scores, n_items)

                if item not in new_recs:
                    counterfactuals[item] = (
                        f"If the group had not interacted with {list(items_to_remove)}, "
                        f"then {item} would not have been suggested."
                    )
                    break
            if item in counterfactuals:
                break

    return counterfactuals


In [ ]:
def print_counterfactual_table(counterfactuals):

    rows = []

    for item, explanation in counterfactuals.items():
      start = explanation.find("[")
      end = explanation.find("]") + 1
      responsible_items = explanation[start:end].strip("[]")

      clean_explanation = explanation.replace("[", "").replace("]", "")

      rows.append([item, responsible_items, clean_explanation])

    print("\n Counterfactual Explanations (Group-Fair Method):\n")
    print(tabulate(
        rows,
        headers=["Recommended\n Item", "Responsible\n Item", "Counterfactual Explanation"],
        tablefmt="grid"
    ))


In [ ]:
if __name__ == "__main__":

    group_users = [1, 2, 3, 4, 5]

    group_scores = compute_group_item_scores(user_item_matrix, group_users)

    counterfactuals = generate_counterfactual_explanations(group_scores, user_item_matrix, group_users)

    print_counterfactual_table(counterfactuals)


 Counterfactual Explanations (Group-Fair Method):

+---------------+---------------+-------------------------------------------------------------------------------------+
|   Recommended |   Responsible | Counterfactual Explanation                                                          |
|          Item |          Item |                                                                                     |
+===============+===============+=====================================================================================+
|           457 |           457 | If the group had not interacted with 457, then 457 would not have been suggested.   |
+---------------+---------------+-------------------------------------------------------------------------------------+
|           608 |           608 | If the group had not interacted with 608, then 608 would not have been suggested.   |
+---------------+---------------+---------------------------------------------------------------------------